# Project – Week 2: Data Cleaning and SQLite Integration
**Dataset:** `original_dataset.csv` (Superstore Dataset from Week 1)

This notebook cleans the Superstore dataset, keeps 5 relevant columns, exports `cleaned_dataset.csv`, and stores the clean data in `../database/project.db`.

In [1]:
import sqlite3
import pandas as pd

df = pd.read_csv("original_dataset.csv", encoding="latin1")
print("Original shape:", df.shape)
df.head()

Original shape: (10800, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2017-152156,11/8/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2.0,0.00,41.9136
1,2,CA-2017-152156,11/8/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3.0,0.00,219.5820
2,3,CA-2017-138688,6/12/2017,6/16/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2.0,0.00,6.8714
3,4,US-2016-108966,10/11/2016,10/18/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5.0,0.45,-383.0310
4,5,US-2016-108966,10/11/2016,10/18/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2.0,0.20,2.5164


## Step 1: Select 5 Relevant Columns
We select 5 columns suitable for filtering, analysis, and visualization:
- **Row ID**: Unique identifier for each order line item (used as primary key).
- **Category**: Product category for grouping and filtering.
- **Region**: Geographic sales region for regional comparison.
- **Sales**: Total sales amount in USD.
- **Profit**: Profit or loss amount in USD.

In [2]:
cols = ["Row ID", "Category", "Region", "Sales", "Profit"]
clean_df = df[cols].copy()
clean_df.columns = ["row_id", "category", "region", "sales", "profit"]
clean_df.head()

,row_id,category,region,sales,profit
0,1,Furniture,South,261.9600,41.9136
1,2,Furniture,South,731.9400,219.5820
2,3,Office Supplies,West,14.6200,6.8714
3,4,Furniture,South,957.5775,-383.0310
4,5,Office Supplies,South,22.3680,2.5164


## Step 2: Clean the Dataset
From Week 1, the CSV contains extra non-order rows at the bottom (with text in `Row ID` and missing values in `Sales`/`Profit`) and duplicate rows. Decisions:
- Remove exact duplicate rows.
- Convert `row_id`, `sales`, and `profit` to numeric; drop rows where any of these or `category`/`region` is missing (removes the stacked non-order rows).
- Strip extra spaces and use consistent title case for `category` and `region`.
- Keep only valid records where `row_id` is an integer and `sales >= 0` (negative `profit` is kept because losses are valid business data).

In [3]:
clean_df = clean_df.drop_duplicates()

for col in ["row_id", "sales", "profit"]:
    clean_df[col] = pd.to_numeric(clean_df[col], errors="coerce")

clean_df = clean_df.dropna(subset=["row_id", "category", "region", "sales", "profit"]).copy()

clean_df["category"] = clean_df["category"].astype(str).str.strip().str.title()
clean_df["region"] = clean_df["region"].astype(str).str.strip().str.title()

clean_df = clean_df[(clean_df["sales"] >= 0) & (clean_df["row_id"] % 1 == 0)].copy()
clean_df["row_id"] = clean_df["row_id"].astype(int)
clean_df = clean_df.drop_duplicates()

print("Cleaned shape:", clean_df.shape)
clean_df.head()

Cleaned shape: (9994, 5)


,row_id,category,region,sales,profit
0,1,Furniture,South,261.9600,41.9136
1,2,Furniture,South,731.9400,219.5820
2,3,Office Supplies,West,14.6200,6.8714
3,4,Furniture,South,957.5775,-383.0310
4,5,Office Supplies,South,22.3680,2.5164


## Step 3: Transform and Export
Round `sales` and `profit` to 2 decimal places, verify there are no missing values or duplicate IDs, and save to `cleaned_dataset.csv`.

In [4]:
clean_df["sales"] = clean_df["sales"].round(2)
clean_df["profit"] = clean_df["profit"].round(2)

assert clean_df["row_id"].is_unique, "row_id must be unique"
assert clean_df.shape[1] == 5, "Must have exactly 5 columns"
assert not clean_df.isna().any().any(), "No missing values allowed"

print("Original records:", len(df))
print("Cleaned records:", len(clean_df))
print("\nMissing values:\n", clean_df.isna().sum())

csv_path = "cleaned_dataset.csv"
clean_df.to_csv(csv_path, index=False)
print("\nSaved:", csv_path)

Original records: 10800
Cleaned records: 9994

Missing values:
 row_id      0
category    0
region      0
sales       0
profit      0
dtype: int64

Saved: cleaned_dataset.csv


## Step 4: Design the Database Table
**Database:** `../database/project.db` | **Table:** `superstore_sales`

| Column | SQLite Type | Constraint |
|---|---|---|
| `row_id` | `INTEGER` | `PRIMARY KEY` |
| `category` | `TEXT` | `NOT NULL` |
| `region` | `TEXT` | `NOT NULL` |
| `sales` | `REAL` | `NOT NULL, CHECK(sales >= 0)` |
| `profit` | `REAL` | `NOT NULL` |

## Step 5: Create the Database and Table
Create `../database/project.db` and define `superstore_sales` using an explicit `CREATE TABLE` statement.

In [5]:
import os
os.makedirs("../database", exist_ok=True)
db_path = "../database/project.db"
create_sql = """
CREATE TABLE IF NOT EXISTS superstore_sales (
    row_id INTEGER PRIMARY KEY,
    category TEXT NOT NULL,
    region TEXT NOT NULL,
    sales REAL NOT NULL CHECK(sales >= 0),
    profit REAL NOT NULL
)
"""

with sqlite3.connect(db_path) as conn:
    conn.execute(create_sql)
print("Database and table ready:", db_path)

Database and table ready: ../database/project.db


## Step 6: Upload the Cleaned Dataset
Clear any previous rows in `superstore_sales` and insert records from `cleaned_dataset.csv`.

In [6]:
upload_df = pd.read_csv("cleaned_dataset.csv")

with sqlite3.connect(db_path) as conn:
    conn.execute("DELETE FROM superstore_sales")
    upload_df.to_sql("superstore_sales", conn, if_exists="append", index=False)

print("Data uploaded successfully:", len(upload_df), "rows")

Data uploaded successfully: 9994 rows


## Step 7: Verify the Uploaded Data
Confirm column definitions, record count match, and sample rows from SQLite.

In [7]:
with sqlite3.connect(db_path) as conn:
    db_count = conn.execute("SELECT COUNT(*) FROM superstore_sales").fetchone()[0]
    schema = conn.execute("PRAGMA table_info(superstore_sales)").fetchall()
    sample = conn.execute("SELECT * FROM superstore_sales LIMIT 5").fetchall()

assert db_count == len(upload_df), "Record count mismatch!"
print("CSV records:", len(upload_df))
print("DB records :", db_count)
print("\nTable schema (cid, name, type, notnull, dflt, pk):")
for col in schema:
    print(" ", col)
print("\nSample 5 rows:")
for row in sample:
    print(" ", row)

CSV records: 9994
DB records : 9994

Table schema (cid, name, type, notnull, dflt, pk):
  (0, 'row_id', 'INTEGER', 0, None, 1)
  (1, 'category', 'TEXT', 1, None, 0)
  (2, 'region', 'TEXT', 1, None, 0)
  (3, 'sales', 'REAL', 1, None, 0)
  (4, 'profit', 'REAL', 1, None, 0)

Sample 5 rows:
  (1, 'Furniture', 'South', 261.96, 41.91)
  (2, 'Furniture', 'South', 731.94, 219.58)
  (3, 'Office Supplies', 'West', 14.62, 6.87)
  (4, 'Furniture', 'South', 957.58, -383.03)
  (5, 'Office Supplies', 'South', 22.37, 2.52)
